In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [9]:
df = pd.read_csv("Emp_Attirition.csv")
df.head(23)

,EmployeeID,Age,Department,JobLevel,YearsAtCompany,MonthlyIncome,JobSatisfaction,WorkLifeBalance,OverTime,DistanceFromHome,PromotionLast5Years,PerformanceRating,TrainingHoursLastYear,Attrition
0,EMP0000,42,Sales,3,9,9866,4,1,No,24.5,Yes,4,19,No
1,EMP0001,36,Sales,3,7,8958,3,3,No,10.0,No,2,13,No
2,EMP0002,44,Sales,4,16,11716,4,3,No,4.0,No,4,13,No
3,EMP0003,53,Finance,4,30,17061,5,3,No,7.1,No,3,20,No
4,EMP0004,35,HR,2,4,5738,2,3,No,8.4,No,2,15,No
5,EMP0005,35,Sales,1,1,3839,2,3,No,3.1,No,2,18,No
6,EMP0006,53,IT,4,31,17377,3,3,No,8.9,No,4,18,No
7,EMP0007,45,Finance,2,5,8463,4,3,No,17.6,No,2,8,No
8,EMP0008,33,HR,2,8,6956,5,3,No,3.6,No,2,7,No
9,EMP0009,43,Engineering,2,4,9383,1,3,Yes,8.6,No,3,10,No


In [10]:
df.drop("EmployeeID", axis=1, inplace=True)

# ENCODING

In [11]:
le = LabelEncoder()
bi_cols = ["OverTime", "PromotionLast5Years", "Attrition"]

for i in bi_cols:
    df[i] = le.fit_transform(df[i])

In [12]:
df

,Age,Department,JobLevel,YearsAtCompany,MonthlyIncome,JobSatisfaction,WorkLifeBalance,OverTime,DistanceFromHome,PromotionLast5Years,PerformanceRating,TrainingHoursLastYear,Attrition
0,42,Sales,3,9,9866,4,1,0,24.5,1,4,19,0
1,36,Sales,3,7,8958,3,3,0,10.0,0,2,13,0
2,44,Sales,4,16,11716,4,3,0,4.0,0,4,13,0
3,53,Finance,4,30,17061,5,3,0,7.1,0,3,20,0
4,35,HR,2,4,5738,2,3,0,8.4,0,2,15,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,51,Marketing,4,28,14510,4,1,0,14.0,0,3,17,0
9996,22,Engineering,1,0,5659,3,3,1,25.3,0,3,19,1
9997,30,Sales,1,0,4223,3,4,1,15.8,0,2,20,0
9998,42,Engineering,2,2,7990,5,3,0,5.7,0,3,25,0


In [13]:
ohe = OneHotEncoder(drop='first',sparse=False)
depart_enc = ohe.fit_transform(df[["Department"]])

In [15]:
dept_cols = ohe.get_feature_names_out(["Department"])
dept_df = pd.DataFrame(depart_enc, columns=dept_cols)

# Remove original Department column
df = df.drop("Department", axis=1)

# Add encoded Department columns
df = pd.concat([df.reset_index(drop=True), dept_df.reset_index(drop=True)], axis=1)

# Train Test Split

In [17]:
X = df.drop("Attrition", axis=1)
y = df["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling

In [18]:
numeric_cols = ["Age","JobLevel","YearsAtCompany","MonthlyIncome",
                "JobSatisfaction","WorkLifeBalance","DistanceFromHome",
                "PerformanceRating","TrainingHoursLastYear"]

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Training

In [19]:
model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

# PREDICTION

In [20]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

# Evaluation

In [21]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

[[1120  693]
 [  64  123]]
              precision    recall  f1-score   support

           0       0.95      0.62      0.75      1813
           1       0.15      0.66      0.25       187

    accuracy                           0.62      2000
   macro avg       0.55      0.64      0.50      2000
weighted avg       0.87      0.62      0.70      2000

ROC AUC: 0.6700714683907961
